# Lab 2C: Completions + Embeddings in Python

**Time**: ~30 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will learn Azure OpenAI chat completions, streaming, and embeddings generation. Practice cosine similarity calculations with generated vectors.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up Cosmos DB and Azure OpenAI client connections.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
EMBEDDINGS_KEY = os.environ.get("EMBEDDINGS_KEY")
DB_NAME = "WorkshopData"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "phi-4-mini-instruct")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "text-embedding-3-small")

for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT", "EMBEDDINGS_ENDPOINT", "EMBEDDINGS_KEY"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Foundry Endpoint:    {FOUNDRY_ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Completions Model:   {COMPLETIONS_MODEL}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

# Embeddings: separate Azure OpenAI resource, API key auth
# (the v1 embeddings surface does not yet support Entra ID).
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=EMBEDDINGS_KEY,
)
print("Foundry chat client + embeddings client initialized")

## Step 1: Chat Completions (STUDENT EXERCISE)

Make a chat completion call with custom options (temperature, max tokens, top_p).

**Expected output**: A response about Cosmos DB partitioning with token usage stats.

In [ ]:
messages = [
    {"role": "system", "content": "You are a data platform expert."},
    {"role": "user", "content": "Explain partitioning in Cosmos DB in 2 sentences."}
]

completion = foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    temperature=0.7,
    max_tokens=200,
    top_p=0.95
)

print(f"Response: {completion.choices[0].message.content}")
print(f"Token usage - Prompt: {completion.usage.prompt_tokens}, Completion: {completion.usage.completion_tokens}")

## Step 2: Streaming Response (STUDENT EXERCISE)

Generate a streaming chat completion response.

In [ ]:
messages = [
    {"role": "user", "content": "List 5 Cosmos DB consistency levels and their use cases."}
]

print("Streaming response: ", end="")
for chunk in foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    stream=True
):
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
print()

## Step 3: Generate Embeddings and Compare (STUDENT EXERCISE)

Generate embeddings for multiple texts and calculate cosine similarity between vectors.

**Expected output**: Embedding dimensions and cosine similarity score.

In [ ]:
import numpy as np


def get_embedding(text: str) -> list[float]:
    """Call the embeddings deployment and return the resulting vector.

    The embeddings model maps each input string to a fixed-size vector
    (1536 dims for text-embedding-3-small) where semantically similar texts
    land near each other in the vector space — this is what makes vector
    search work.
    """
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


texts = [
    "Azure Cosmos DB is globally distributed.",
    "Microsoft Azure is a cloud platform.",
    "Cosmos DB vector search supports semantic similarity."
]

embeddings_list = []

print("Generating embeddings:")
for text in texts:
    embedding = get_embedding(text)
    embeddings_list.append((text, embedding))
    print(f"  {text[:40]}... dim={len(embedding)}")

# Compare docs 1 and 3 — both are about Cosmos DB, so we expect a high
# similarity score relative to doc 2 (a generic Azure statement).
# Cosine similarity is dot(a, b) / (|a| * |b|) — numpy gives us the primitives.
vec_a = np.array(embeddings_list[0][1])
vec_b = np.array(embeddings_list[2][1])
cosine_similarity = float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))

print(f"\nCosine similarity (docs 1 vs 3): {cosine_similarity:.4f}")
print("Note: Higher value = more similar (range: -1 to 1)")

print("\n=== Lab Complete ===")
print("You have completed the Completions + Embeddings exercise in Python. You:")
print("- Made a chat completion call")
print("- Generated a streaming response")
print("- Generated embeddings for multiple texts")
print("- Calculated cosine similarity between vectors")